In [1]:
import numpy as np

Defining users and channels

In [2]:
n_users = 50      # secondary users
n_channels = 60     # available spectrum bands

Getting SINR from random

In [3]:
# Simulate channel gains randomly (in reality this comes from pathloss model)
# SINR[i][j] = signal quality if user i uses channel j
np.random.seed(67)
channel_gain = np.random.uniform(0.1, 1.0, (n_users, n_channels)) # to get SINR which gets throughput
noise = 0.1

SINR = channel_gain / noise  # simplified, no inter-user interference yet

SINR

array([[5.91266287, 8.7297095 , 7.17303328, ..., 3.15823553, 7.08705175,
        4.86264092],
       [3.02554615, 2.36651987, 5.08196692, ..., 2.631924  , 4.69246185,
        6.87450396],
       [1.7659955 , 6.28816609, 7.54302947, ..., 5.64675291, 6.57603255,
        2.98752236],
       ...,
       [3.60699369, 4.4563863 , 5.51891613, ..., 3.06195476, 6.62259474,
        2.50503304],
       [9.76246376, 3.99055316, 8.36853942, ..., 2.00138352, 2.00760341,
        6.28659795],
       [8.11545745, 6.11541862, 7.52250623, ..., 6.57975057, 8.38636869,
        4.66045815]], shape=(50, 60))

Setting up Primary Users

In [4]:
n = n_channels          # array size
k = 7           # number of ones

arr = np.zeros(n, dtype=int)
arr[np.random.choice(n, k, replace=False)] = 1


# Primary user occupancy: PU[j] = 1 means channel j is occupied by primary user
PU_occupied = arr  # eg channels 1 and 3 are taken for [1 0 0 1 0 0 ...]
PU_occupied

array([0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

Fitness function which gets the throughput using SINR and adds penalty

In [5]:
def fitness(x):
    channels = np.clip(np.round(x).astype(int), 0, n_channels - 1)
    
    throughput = 0
    penalty = 0
    for user in range(n_users):
        ch = channels[user]
        if PU_occupied[ch]:
            penalty += 1000  # hard penalty for interfering with primary user
        else:
            throughput += np.log2(1 + SINR[user, ch])
    
    # secondary user collision penalty
    for ch in range(n_channels):
        users_on_ch = np.sum(channels == ch)
        if users_on_ch > 1:
            penalty += (users_on_ch - 1) * 1000
    
    return -throughput + penalty




PSO algorithm

In [6]:

def pso(f, N, S, a, b, b_hat, c, D):

    
    x = np.random.uniform(0, S, size=(N, D))
    v = np.random.normal(size=(N, D))
    p = x.copy()
    p_hat = x[np.argmin(np.apply_along_axis(f, 1, x))]

    for i in range(1000):
        if i%100 == 0:
            print("doing...",i)
        r, r_hat = np.random.uniform(0, 1, (2, N, D))

        v = a*v + b*r*(p-x) + b_hat*r_hat*(p_hat-x)
        x = x + c*v
        

        for i in range(N):
            if f(x[i]) < f(p[i]):
                p[i] = x[i]
            if f(p[i]) < f(p_hat):
                p_hat = p[i]

    return p_hat

Calling PSO and printing throughput

In [7]:
result = pso(
    f=fitness,
    N=150,           # particles
    S=n_channels-1, # search space [0, n_channels-1]
    a=0.7,
    b=1.5,
    b_hat=1.5,
    c=0.1,
    D=n_users       # dimensions = number of users
)


best_assignment = np.clip(np.round(result).astype(int), 0, n_channels-1)

actual_throughput = 0
for user in range(n_users):
    ch = best_assignment[user]
    if not PU_occupied[ch]:
        actual_throughput += np.log2(1 + SINR[user, ch])

print("Channel assignment:", best_assignment)
print("Throughput:", actual_throughput)

doing... 0
doing... 100
doing... 200
doing... 300
doing... 400
doing... 500
doing... 600
doing... 700
doing... 800
doing... 900
Channel assignment: [11 35 23 25  8 32 54 30 21 10 50 47 37 50  6 12 27 16 51 44 26 46 19 14
 33 29 52 55 23 42 41 17 43 31 12 15 12 33 18 50 56 35 28 47 36  3 25 53
 22 38]
Throughput: 118.26275614374822


In [8]:
best_assignment

array([11, 35, 23, 25,  8, 32, 54, 30, 21, 10, 50, 47, 37, 50,  6, 12, 27,
       16, 51, 44, 26, 46, 19, 14, 33, 29, 52, 55, 23, 42, 41, 17, 43, 31,
       12, 15, 12, 33, 18, 50, 56, 35, 28, 47, 36,  3, 25, 53, 22, 38])

In [9]:
actual_throughput

np.float64(118.26275614374822)

In [10]:
def qpso(f, N, beta_start, beta_end, n_iter, D, low, high):
    
    # Initialize positions
    x = np.random.uniform(low, high, size=(N, D))
    
    # Personal bests
    p = x.copy()
    p_fitness = np.array([f(p[i]) for i in range(N)])
    
    # Global best
    g = p[np.argmin(p_fitness)].copy()
    
    for t in range(n_iter):
        
        # Beta decreases linearly from beta_start to beta_end
        beta = beta_start - (beta_start - beta_end) * t / n_iter
        
        # Mean best position — average of all personal bests
        mbest = np.mean(p, axis=0)
        
        for i in range(N):
            if i%100 == 0:
                print("doing...",i)
            # Local attractor — random point between personal best and global best
            phi = np.random.uniform(0, 1, D)
            p_local = phi * p[i] + (1 - phi) * g
            
            # Sample new position from Laplace distribution
            u = np.random.uniform(0, 1, D)
            sign = np.where(np.random.rand(D) < 0.5, 1, -1)
            x[i] = p_local + sign * beta * np.abs(mbest - x[i]) * np.log(1/u)
            
            # Clip to valid range
            x[i] = np.clip(x[i], low, high)
            
            # Update personal best
            if f(x[i]) < p_fitness[i]:
                p[i] = x[i].copy()
                p_fitness[i] = f(x[i])
            
            # Update global best
            if p_fitness[i] < f(g):
                g = p[i].copy()
    
    return g


result_qpso = qpso(
    f=fitness,
    N=150,
    beta_start=1.0,
    beta_end=0.5,
    n_iter=1000,
    D=n_users,
    low=0,
    high=n_channels-1
)

best_assignment = np.clip(np.round(result_qpso).astype(int), 0, n_channels-1)
actual_throughput = 0
for user in range(n_users):
    ch = best_assignment[user]
    if not PU_occupied[ch]:
        actual_throughput += np.log2(1 + SINR[user, ch])

print("Channel assignment:", best_assignment)
print("Throughput:", actual_throughput)

doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing... 100
doing... 0
doing